# Chapter 2 — Data Manipulation: Network Traffic Data Cleaning

## 📋 Latihan Soal

### Dataset
- **Mirai Botnet Dataset** — https://huggingface.co/datasets/bornpresident/mirai_botnet
- 1.083.206 baris, 48 kolom
- Sample 5000 baris untuk EDA


In [ ]:
# ============================================
# SETUP: Load Mirai Botnet Dataset
# ============================================
# Dataset: https://huggingface.co/datasets/bornpresident/mirai_botnet
# 1.083.206 baris, 48 kolom, 4 kelas traffic
# Label: BenignTraffic, Mirai-greeth_flood, Mirai-udpplain, Mirai-greip_flood
from datasets import load_dataset
import pandas as pd
import numpy as np

# Load dataset dari HuggingFace
ds = load_dataset('bornpresident/mirai_botnet')
df = ds['train'].to_pandas()

# Rename kolom agar lebih mudah dipakai (ganti spasi dengan underscore)
df.columns = [c.replace(' ', '_') for c in df.columns]

# Rename kolom 'label' jadi 'traffic_type' agar konsisten
df = df.rename(columns={'label': 'traffic_type'})

# Konversi is_benign ke format Benign/Malicious
df['traffic_type'] = df['traffic_type'].apply(
    lambda x: 'Benign' if x == 'BenignTraffic' else 'Malicious'
)

# Ambil sample 5000 baris untuk performa notebook
# (dataset penuh 1M+ baris — sample cukup untuk EDA)
np.random.seed(42)
df = df.sample(n=5000, random_state=42).reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Traffic types:\n{df['traffic_type'].value_counts()}")
print(f"\nKolom ({len(df.columns)} total):")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col} ({df[col].dtype})")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")


---
## Soal 1: Grouping — Agregasi per Traffic Type

Kelompokkan data berdasarkan `traffic_type` dan `Protocol_Type`. Hitung mean, std, dan count untuk setiap fitur numerik. Bandingkan statistik antara traffic Benign vs Malicious.


In [ ]:

# Grouping by traffic_type
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
grouped = df.groupby('traffic_type')[numeric_cols].agg(['mean', 'std', 'count'])
print("=== Statistik per Traffic Type ===")
print(grouped.round(4))

# Grouping by Protocol_Type
proto_groups = df.groupby('Protocol_Type')[['Rate', 'Tot_size', 'Duration']].agg(['mean', 'max', 'min'])
print("\n=== Agregasi per Protocol Type ===")
print(proto_groups.round(4))


---
## Soal 2: Appending — Gabungkan Subset Data

Pisahkan data Benign dan Malicious, kemudian append kembali menjadi satu DataFrame. Cek konsistensi index dan jumlah baris.


In [ ]:

# Split dataset
df_benign = df[df['traffic_type'] == 'Benign'].copy()
df_malicious = df[df['traffic_type'] == 'Malicious'].copy()

# Append: gabungkan kembali
df_appended = pd.concat([df_benign, df_malicious], ignore_index=True)
print(f"Benign: {len(df_benign)} rows")
print(f"Malicious: {len(df_malicious)} rows")
print(f"Appended: {len(df_appended)} rows")
print(f"Index reset: {df_appended.index.is_monotonic_increasing}")


---
## Soal 3: Concatenating — Gabungkan dan Dedup

Concatenate data Benign dengan data Malicious beberapa kali sehingga ada duplikat. Cek total baris sebelum dan sesudah deduplication.


In [ ]:

# Concatenate: gabungkan data per jenis serangan
mirai_types = df[df['traffic_type'] == 'Malicious']['traffic_type'].unique()
frames = [df[df['traffic_type'] == 'Benign']]
for t in mirai_types:
    frames.append(df[df['traffic_type'] == 'Malicious'])

df_concat = pd.concat(frames, ignore_index=True)
print(f"Original: {len(df)} rows")
print(f"After concat: {len(df_concat)} rows")
print(f"Duplikat: {df_concat.duplicated().sum()} rows")

# Dedup
df_dedup = df_concat.drop_duplicates()
print(f"Setelah dedup: {len(df_dedup)} rows")


---
## Soal 4: Merging — Join dengan Threat Intelligence

Buat tabel threat intelligence yang mapping `Protocol_Type` ke nama protokol, risk level, dan jenis serangan. Merge dengan dataset utama.


In [ ]:

# Buat threat intelligence table
threat_intel = pd.DataFrame({
    'Protocol_Type': [6, 17, 47, 1, 2],
    'protocol_name': ['TCP', 'UDP', 'GRE', 'ICMP', 'IGMP'],
    'risk_level': ['Medium', 'High', 'Critical', 'Low', 'Medium'],
    'common_attack': ['SYN Flood', 'UDP Flood', 'GRE Tunnel Flood', 'Ping Flood', 'IGMP Flood'],
})

# Merge berdasarkan Protocol_Type
df_merged = df.merge(threat_intel, on='Protocol_Type', how='left')
print(f"Shape setelah merge: {df_merged.shape}")
print(f"Kolom baru: {[c for c in df_merged.columns if c not in df.columns]}")
print(f"\nDistribusi risk level:")
print(df_merged['risk_level'].value_counts())


---
## Soal 5: Sorting — Urutkan Berdasarkan Risk & Rate

Urutkan data berdasarkan risk level (Critical > High > Medium > Low) sebagai primary sort, kemudian `Rate` descending sebagai secondary sort.


In [ ]:

# Sorting: Rate descending, Duration ascending
risk_order = {'Critical': 0, 'High': 1, 'Medium': 2, 'Low': 3, None: 4}
df_sorted = df_merged.copy()
df_sorted['risk_order'] = df_sorted['risk_level'].map(risk_order).fillna(4)
df_sorted = df_sorted.sort_values(['risk_order', 'Rate'], ascending=[True, False])
df_sorted = df_sorted.drop(columns=['risk_order'])

print("=== Top 10 Highest Rate (sorted by risk) ===")
print(df_sorted[['traffic_type', 'Protocol_Type', 'Rate', 'risk_level']].head(10).to_string())


---
## Soal 6: Categorising — Kategori Traffic Level

Kategorikan `Rate` ke dalam 4 level: Low (<50), Medium (50-200), High (200-500), Critical (>500). Buat kolom baru `traffic_level`.


In [ ]:

# Categorize Rate ke dalam level
bins = [0, 50, 200, 500, float('inf')]
labels = ['Low', 'Medium', 'High', 'Critical']
df_merged['traffic_level'] = pd.cut(df_merged['Rate'], bins=bins, labels=labels)

print("=== Distribusi Traffic Level ===")
print(df_merged['traffic_level'].value_counts())
print(f"\n=== Rate per Level ===")
print(df_merged.groupby('traffic_level')['Rate'].agg(['min', 'mean', 'max', 'count']).round(2))


---
## Soal 8: Dropping Rows & Columns

1. Drop kolom yang tidak diperlukan (ece_flag_number, cwr_flag_number, urg_count, rst_count)
2. Drop baris yang `Rate`-nya NaN


In [ ]:

# Drop kolom yang sudah tidak diperlukan
cols_to_drop = ['ece_flag_number', 'cwr_flag_number', 'urg_count', 'rst_count']
cols_to_drop = [c for c in cols_to_drop if c in df_merged.columns]
df_dropped = df_merged.drop(columns=cols_to_drop)
print(f"Kolom sebelum drop: {df_merged.shape[1]}")
print(f"Kolom setelah drop: {df_dropped.shape[1]}")

# Drop baris dengan Rate NaN
before = len(df_dropped)
df_dropped = df_dropped.dropna(subset=['Rate'])
print(f"Baris sebelum drop NaN: {before}")
print(f"Baris setelah drop NaN: {len(df_dropped)}")
print(f"Baris dihapus: {before - len(df_dropped)}")


---
## Soal 9: Changing Data Format

1. Konversi `Protocol_Type` ke integer
2. Buat kolom baru `Rate_category` dengan kategori: Rendah, Sedang, Tinggi, Sangat Tinggi
3. Konversi `traffic_level` ke string


In [ ]:

# Konversi tipe data
df_merged['Protocol_Type'] = df_merged['Protocol_Type'].astype(int)
df_merged['traffic_level'] = df_merged['traffic_level'].astype(str)

# Buat kolom baru dari existing columns
df_merged['Rate_category'] = pd.cut(
    df_merged['Rate'],
    bins=[0, 100, 500, 1000, float('inf')],
    labels=['Rendah', 'Sedang', 'Tinggi', 'Sangat Tinggi']
).astype(str)

print("=== Tipe Data Setelah Konversi ===")
print(df_merged[['Protocol_Type', 'traffic_level', 'Rate_category']].dtypes)
print(f"\n=== Distribusi Rate Category ===")
print(df_merged['Rate_category'].value_counts())


---
## Soal 10: Replacing Data

1. Ganti `risk_level` yang NaN menjadi 'Unknown'
2. Ganti label 'Benign' menjadi 'Normal' dan 'Malicious' menjadi 'Attack'


In [ ]:

# Replace nilai risk_level yang NaN
df_merged['risk_level'] = df_merged['risk_level'].fillna('Unknown')
print(f"Risk level setelah replace NaN:")
print(df_merged['risk_level'].value_counts())

# Replace traffic_type label
df_merged['traffic_type'] = df_merged['traffic_type'].replace({
    'Benign': 'Normal',
    'Malicious': 'Attack'
})
print(f"\nTraffic type setelah replace:")
print(df_merged['traffic_type'].value_counts())


---
## Soal 11: Dealing with Missing Values

Handle missing values dengan 3 metode berbeda:
1. **Forward Fill** — Isi dengan nilai sebelumnya
2. **Mean Imputation** — Isi dengan rata-rata per kolom
3. **Interpolasi** — Interpolasi linear


In [ ]:

# Cek missing values
print("=== Missing Values per Kolom ===")
missing = df_merged.isnull().sum()
print(missing[missing > 0])

# Metode 1: Forward Fill (untuk time series)
df_ffill = df_merged.ffill()
print(f"\nMetode 1 (Forward Fill): {df_ffill.isnull().sum().sum()} missing tersisa")

# Metode 2: Mean Imputation
df_mean = df_merged.copy()
numeric_cols = df_mean.select_dtypes(include=[np.number]).columns
df_mean[numeric_cols] = df_mean[numeric_cols].fillna(df_mean[numeric_cols].mean())
print(f"Metode 2 (Mean Imputation): {df_mean.isnull().sum().sum()} missing tersisa")

# Metode 3: Interpolasi
df_interp = df_merged.copy()
df_interp[numeric_cols] = df_interp[numeric_cols].interpolate()
print(f"Metode 3 (Interpolasi): {df_interp.isnull().sum().sum()} missing tersisa")
